# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

> **DRAFT prepared with an AI assistant, per this card's own instruction to load `writing-data-contracts` + `flyrank/flyrank-data`.** Every cell below is real, correct methodology and real, runnable code — but it has NOT been executed against the live Hugging Face warehouse (no token/network access in the drafting environment). Simon: run every cell yourself in Colab with your own HF_TOKEN secret, confirm the real outputs match the claims below, fix anything that doesn't, and only then commit. Do not submit fabricated or assumed outputs — this is the exact discipline the whole internship has been building toward.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. The contract, in plain words (5 answers)

**Lane:** Freestyle — Diagnosis-First Content Triage (synthesizing Lane 2: Refresh/Content Opportunity Scoring, and Lane 4: CTR/Engagement Opportunity Scoring). See ML-02/ML-03 for the full framing.

1. **What one row means:** one row = one (`client_hash_id`, `content_hash_id`, `report_date`) record from `fact_content_daily_performance` — i.e., one content item's measured performance on one single day. This matches the table's own documented grain (`report_date + client_hash_id + content_hash_id`), so nothing needs to be invented or aggregated for this contract check.

2. **Which table(s):** `fact_content_daily_performance` (the daily fact table) as the primary source, joined to `dim_content` (for `main_intent`, `content_type`, and other content metadata) and `dim_clients` (for `gsc_data_start` / `ga4_data_start`, to respect per-client history depth).

3. **Time window:** a single mid-panel calendar month, `month=2026-03` (2026-03-01 through 2026-03-31), per the card's explicit warning to never develop label logic on the `_sample` table (which is only the final month, June 2026, and would leak the future outcome window into development).

4. **What I'd predict or rank (label/proxy) — TWO separate targets, kept deliberately apart:**

   - **Target A, `is_declining`:** predicts whether a page's *impressions* are trending down, built from `trend_direction` (itself comparing `impressions_last_30d` vs `impressions_prev_30d`). This is a proxy because a single 30-day-vs-30-day comparison might just be a temporary blip or noise, not a confirmed, lasting decline.
   - **Target B, `diagnosis`:** predicts *why* a flagged page is struggling — genuine_decline / likely_serp_answered / ctr_fixable / stable_or_improving — built by comparing how *impressions and clicks move together*. This is a separate proxy because even a matching pattern (e.g., impressions steady, clicks falling) is an inference about the cause, not a confirmed fact (the real query/URL is never seen, so no diagnosis can be firmly proven — see the SERP-Interception Diagnosis Framework for the full reasoning).

   **This single month cannot yet compute Target A** (it needs 60+ days of history: the current 30 days plus the prior 30) — this contract only proves the raw daily columns needed to build both targets later actually exist and are populated. See Section 4 for this named as an explicit limitation.

5. **One thing deliberately excluded:** any FlyRank product-decision column (`health_score`, `priority_score`, `action_type`) if any such column is ever visible anywhere in the warehouse — excluded because it would encode a prior decision already made by the existing system, not a raw observed signal, and using it as a feature would produce a circular result (see `SKILL.md/hunting-leakage-and-validating`). `keyword_hash_id` and `url_hash_id` are also excluded as model features — context/grouping only, per the join rules in `ml-intern-dataset-and-lane-guide.md`.

In [ ]:
# Setup — run this in Colab with HF_TOKEN stored as a Secret (never pasted in a cell; this repo is public).
%pip -q install duckdb
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')  # Secrets panel, NOT a pasted string
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Table shapes per the Hugging Face 'Files' tab -- confirm folder-vs-single-file before running.
DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

# Cheap metadata-only sanity check before touching real data.
print(con.sql(f"SELECT COUNT(*) AS n, MIN(report_date) AS min_d, MAX(report_date) AS max_d FROM {DAILY} WHERE month='2026-03'").df())

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (daily) | Feature | Raw, observed, known as soon as that day's data lands -- available at any later decision moment |
| `ga4_sessions`, `ga4_data_available` | Feature / availability flag | Observed GA4 measurement plus its own honesty flag (see Section 3, query 3) |
| `main_intent`, `content_type` (from `dim_content`) | Feature | Stable content metadata, known long before any given day's performance |
| `report_date` (day of week, recency) | Feature | Purely calendar-derived -- knowable even in advance |
| `trend_direction`, `trend_pct` (if present at this grain) | Label-source -- NEVER a feature | This is where the eventual label comes from; using it as an input would be circular |
| `content_hash_id`, `client_hash_id`, `keyword_hash_id`, `url_hash_id` | Context | Grouping/joining/splitting only -- carry no real-world meaning and must never be model inputs |
| `health_score`, `priority_score`, `action_type` (if ever visible) | Excluded | FlyRank's own prior decision, not a raw signal -- would leak the existing system's answer back into the model |

## 3. Verify it with queries (grain, counts, missing values, windows)

Three required checks, run against `month='2026-03'` only.

In [ ]:
# Query 1 -- GRAIN CHECK: one row really is one (client, content, day). Expect ZERO rows back.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {DAILY}
    WHERE month = '2026-03'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print('Grain violations found (should be empty):')
print(grain_check)

In [ ]:
# Query 2 -- ROW COUNT + DATE SPAN for this slice.
count_and_span = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {DAILY}
    WHERE month = '2026-03'
""").df()
print(count_and_span)

In [ ]:
# Query 3 -- AVAILABILITY, filtered with IS TRUE (NOT '= TRUE' and NOT 'NOT flag' -- NULL is neither).
total_rows = con.sql(f"SELECT COUNT(*) AS n FROM {DAILY} WHERE month = '2026-03'").df()['n'][0]
available_rows = con.sql(f"""
    SELECT COUNT(*) AS n
    FROM {DAILY}
    WHERE month = '2026-03' AND ga4_data_available IS TRUE
""").df()['n'][0]

print(f"Total rows this month: {total_rows}")
print(f"Rows with GA4 data actually available (IS TRUE, not just truthy): {available_rows}")
print(f"Share surviving the availability filter: {available_rows/total_rows:.1%}")
# If this ratio looks suspiciously close to 100% or 0%, re-check for NULLs before trusting it --
# per flyrank-data: millions of rows have this flag NULL, not FALSE.

## 4. Five features (max) + the leakage trap

**Five features, each tagged with why it's knowable at the decision moment:**

1. `gsc_impressions` (daily) -- knowable because it's a closed, already-measured fact for a day that has passed.
2. `gsc_clicks` (daily) -- same reasoning; a completed day's measurement.
3. `gsc_avg_position` (daily) -- same; FlyRank/Google already recorded this ranking outcome for that day.
4. `ga4_data_available` (flag) -- knowable immediately from the row itself; tells you whether to trust GA4-derived columns for that specific day, before you use any of them.
5. `day_of_week` (derived from `report_date`) -- purely calendar arithmetic; knowable even in advance of the day itself.

**The trap, performed on purpose:** build a toy label, add one label-derived (leaky) feature, watch the score jump toward perfect, then delete it and keep the honest number.

In [ ]:
# Pull a small, real feature frame for this month (adjust column names to match the actual schema
# once you've run Section 1's setup cell and can see the real columns with a DESCRIBE).
features_df = con.sql(f"""
    SELECT
        d.client_hash_id, d.content_hash_id, d.report_date,
        d.gsc_impressions, d.gsc_clicks, d.gsc_avg_position, d.ga4_data_available,
        dayofweek(d.report_date) AS day_of_week
    FROM {DAILY} d
    WHERE d.month = '2026-03'
""").df()
print(features_df.shape)
print(features_df.head())

# --- THE TRAP ---
# Toy label: is this row's impressions below this month's median? (a stand-in for 'underperforming')
median_impr = features_df['gsc_impressions'].median()
features_df['toy_label_low_impressions'] = (features_df['gsc_impressions'] < median_impr).astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

honest_X = features_df[['gsc_avg_position', 'day_of_week']].fillna(0)
leaky_X  = features_df[['gsc_avg_position', 'day_of_week', 'gsc_impressions']].fillna(0)  # <-- the leak
y = features_df['toy_label_low_impressions']

for name, X in [('HONEST (no leak)', honest_X), ('LEAKY (label-derived feature included)', leaky_X)]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    score = model.score(X_te, y_te)
    print(f"{name}: accuracy = {score:.3f}")

# Expect: the LEAKY version jumps toward ~1.0, because gsc_impressions is literally what defined
# the label. Delete gsc_impressions from the feature set (use honest_X going forward) and KEEP
# the honest, lower number -- that is the real, trustworthy result.

## Data limits (one named limitation)

**This single-month slice cannot yet compute the real trend label.** The eventual ranking label needs a last-30-day vs prior-30-day comparison (60 days of history), but this contract deliberately checks only one mid-panel month for grain/availability/feature-presence. Building the real label requires pulling a second, adjacent month and constructing the rolling comparison in ML-05 -- this notebook proves the raw ingredients exist and are trustworthy, not that the label is ready yet.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.